# Phase 1: Data Processing
This notebook extracts and structures text from Project Gutenberg HTML files.

### Environment Setup
If you encounter `ModuleNotFoundError`, run the following cell to install necessary libraries in your current Jupyter environment.

In [1]:
import sys
!{sys.executable} -m pip install beautifulsoup4 spacy nltk lxml
!{sys.executable} -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


### Imports & Initialization

In [2]:
import os
import json
import spacy
import nltk
import string
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

# Ensure NLTK data is available
def ensure_nltk_data():
    try:
        nltk.data.find('tokenizers/punkt')
        nltk.data.find('tokenizers/punkt_tab')
    except LookupError:
        nltk.download('punkt')
        nltk.download('punkt_tab')
    try:
        nltk.data.find('corpora/stopwords')
    except LookupError:
        nltk.download('stopwords')
    try:
        nltk.download('averaged_perceptron_tagger')
        nltk.download('averaged_perceptron_tagger_eng')
        nltk.download('maxent_ne_chunker')
        nltk.download('words')
    except:
        pass

# Load spaCy model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("spaCy model 'en_core_web_sm' not found. Run the setup cell above.")
    nlp = None

ensure_nltk_data()

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/manikantapotla/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/manikantapotla/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /Users/manikantapotla/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     /Users/manikantapotla/nltk_data...
[nltk_data]   Package words is already up-to-date!


In [3]:
def clean_html_content(html_path):
    """
    Reads HTML file, removes boilerplate, and extracts clean paragraphs.
    """
    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'lxml')
    
    # Remove common irrelevant tags
    for tag in soup(['script', 'style', 'header', 'footer']):
        tag.decompose()
    
    # Remove Gutenberg-specific boilerplate
    for tag in soup.find_all(id=['pg-header', 'pg-footer']):
        tag.decompose()
    for tag in soup.find_all(class_=['pg-boilerplate', 'pgheader', 'pgfooter']):
        tag.decompose()

    # Extract clean paragraphs
    paragraphs = []
    for p in soup.find_all('p'):
        text = p.get_text().strip()
        if len(text) > 20: 
            paragraphs.append(text)
    
    full_text = " ".join(paragraphs)
    return full_text, paragraphs

In [4]:
def process_nlp(text, paragraphs):
    """
    Extracts characters, keywords, and sentences.
    """
    sentences = sent_tokenize(text)
    
    # Character extraction
    characters_set = set()
    if nlp:
        # Limit text for NER performance if file is huge
        test_text = text[:100000]
        doc = nlp(test_text)
        for ent in doc.ents:
            if ent.label_ == "PERSON" and len(ent.text) > 2:
                characters_set.add(ent.text.strip().replace('\n', ' '))
    
    # Keyword extraction
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(text.lower())
    keywords_clean = [w for w in words if w.isalpha() and w not in stop_words and len(w) > 3]
    
    freq_dist = nltk.FreqDist(keywords_clean)
    top_keywords = [word for word, count in freq_dist.most_common(20)]
    
    return sorted(list(characters_set)), top_keywords, sentences

In [5]:
dataset_dir = "../dataset"
output_dir = "../backend/data_processed"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

files = [f for f in os.listdir(dataset_dir) if f.endswith(".html")]

for filename in files:
    print(f"--- Processing {filename} ---")
    file_path = os.path.join(dataset_dir, filename)
    
    try:
        clean_text, paragraphs = clean_html_content(file_path)
        characters, keywords, sentences = process_nlp(clean_text, paragraphs)
        
        output_data = {
            "source_file": filename,
            "text": clean_text,
            "characters": characters,
            "keywords": keywords,
            "sentences": sentences[:1000]
        }
        
        output_path = os.path.join(output_dir, filename.replace(".html", ".json"))
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(output_data, f, indent=2)
        
        print(f"Done. Characters found: {len(characters)}")
    except Exception as e:
        print(f"Error: {e}")

--- Processing pg22217-images.html ---
Done. Characters found: 67
--- Processing pg14499-images.html ---
Done. Characters found: 120
--- Processing pg15474-images.html ---
Done. Characters found: 253
--- Processing pg7128-images.html ---
Done. Characters found: 74
--- Processing pg15586-images.html ---
Done. Characters found: 106
--- Processing pg3310-images.html ---
Done. Characters found: 180
--- Processing pg24461-images.html ---
Done. Characters found: 41
--- Processing pg24869-images.html ---
Done. Characters found: 399
--- Processing pg1470-images.html ---
Done. Characters found: 207
--- Processing pg11212-images.html ---
Done. Characters found: 70
--- Processing pg20847-images.html ---
Done. Characters found: 111
--- Processing pg8649-images.html ---
Done. Characters found: 92
